# Operator Compiler Tutorial

Welcome to the **Operator Compiler Tutorial**!

This interactive notebook will guide you through the fundamentals of **performance engineering** using the XTC compiler, a research platform for optimizing AI operators.

By the end of this notebook, you will understand how to:

- Define computational graphs with XTC
- Compile and evaluate operator performance
- Explore the scheduling space to find optimal configurations

## Define your first graph

In XTC, computations are represented as **dataflow graphs**. A graph consists of:
- **Nodes** representing operations each implementing a specific computation (Operators)
- **Edges** representing data dependencies between operations (through Tensors)

Where:
- **Tensors** are multi-dimensional arrays that hold data
- **Operators** are tensor operations (e.g., matrix multiplication, convolution)

Let us start by creating a simple matrix multiplication graph. Matrix multiplication (matmul) computes $C = A \times B$ where:
- $A$ is an $I \times K$ matrix
- $B$ is a $K \times J$ matrix
- $C$ is the resulting $I \times J$ matrix

**Practice.** The code below is editable and the output (i.e. the serialized graph) is dynamically computed.

1. Try modifying the dimensions or data type (float32, float64) to see how the graph changes!
2. Add a ReLU activation after the matrix multiplication. Hint: `O.matmul()` returns a tensor that can be passed to another operator. Using ReLu: `R = O.relu(inp, name)`.

In [1]:
import xtc.graphs.xtc.op as O
from xtc.graphs.xtc.graph import XTCGraph

def matmul_graph(I: int, J: int, K: int, dtype: str) -> XTCGraph:
   """Create a graph computing C = A @ B."""
   a = O.tensor((I, K), dtype, name="A")
   b = O.tensor((K, J), dtype, name="B")
   with O.graph(name="matmul") as gb:
     C = O.matmul(a, b, name="C")
   return gb.graph

I, J, K, dtype = 16, 1000, 512, "float32"
graph = matmul_graph(I=I,J=J,K=K,dtype=dtype)

print(graph)

graph:
  name: matmul
  inputs:
  - %0 : 16x512xfloat32
  - %1 : 512x1000xfloat32
  outputs:
  - %2 : 16x1000xfloat32
  nodes:
  - %2: matmul(%0, %1) {name = 'C'} : [16x512xfloat32, 512x1000xfloat32] -> [16x1000xfloat32]



## Compile and Evaluate

Now that we have a graph, we can compile it and measure its baseline performance (without any optimization).

The compilation pipeline in XTC follows these steps:
1. **Create a Backend**: In XTC, the backend corresponds to an existing framework such as MLIR or TVM that, given a schedule, can generate the code for a specific target
2. **Get a Scheduler**: In XTC, a scheduler is a builder that creates a schedule. Even without optimizations, we need a scheduler to get a default loop structure.
3. **Compile**: Generate executable code
4. **Evaluate**: Run the compiled code and measure performance

**Practice.** The script below compiles the matmul graph without any optimization and displays both the generated code (if you unroll the accordion) and its performance on chip.

1. Use the radio buttons to select optionally an output (IR, generated assembly or equivalent optimized C code)
2. *Inspect the generated code.* Look at the Source IR, Transformed IR, aAssembly. Are you able to identify the different loops i, j, k ?
3. *Observe the performance.* In your opinion, why is it so poor?

**Note:** Performance is measured as a percentage of peak (the theoretical FLOP/s of the CPU). The latter is by default approximated by running a benchmark, but this technique comes with (little) noise. If you already know the peak perf of your machine in GFlops/s, you can set the variable `peak_flops` yourself.

In [6]:
import ipy
ipy.display_nradio("print", "Print options", "none", "source IR", "transformed IR", "Assemnly", "C")

RadioButtons(description='Print options', options=(('none', 'none'), ('source IR', 'source IR'), ('transformed…

In [10]:
import ipy
import xtc.graphs.xtc.op as O
from xtc.graphs.xtc.graph import XTCGraph
from xtc.backends.tvm import Backend
from xtc.runtimes.host import HostRuntime
from pathlib import Path

def matmul_graph(I: int, J: int, K: int, dtype: str) -> XTCGraph:
   """Create a graph computing C = A @ B."""
   a = O.tensor((I, K), dtype, name="A")
   b = O.tensor((K, J), dtype, name="B")
   with O.graph(name="matmul") as gb:
     C = O.matmul(a, b, name="C")
   return gb.graph

# Problem setup
I, J, K, dtype = 16, 1000, 512, "float32"
base = "matmul_naive"
graph = matmul_graph(I, J, K, dtype)
backend = Backend(graph)

# Compile (no transformations, just default loop structure)
scheduler = backend.get_scheduler()
schedule = scheduler.schedule()

print_opts = dict(
    print_source_ir=ipy.nradio_value("print", "source IR"),
    print_transformed_ir=ipy.nradio_value("print", "transformed IR"),
    print_assembly=ipy.nradio_value("print", "Assembly"),
    emit_c=ipy.nradio_value("print", "C"),
)
compiler = backend.get_compiler(dump_file="matmul_naive", shared_lib=True, **print_opts)
module = compiler.compile(schedule)

if ipy.nradio_value("print", "C"):
    print(Path(f"{base}.c").read_text())

# Evaluate and display results
peak_flops = HostRuntime.get().evaluate_flops(dtype)
evaluator = module.get_evaluator()
results, _, _ = evaluator.evaluate()
perf = (I * J * K) / min(results) / peak_flops * 100

print(f"\nPeak perfromance: {perf:.2f} %")

// tvm target: c -keys=arch -march=generic -mcpu=generic
#define TVM_EXPORTS
#include "tvm/runtime/c_runtime_api.h"
#include "tvm/runtime/c_backend_api.h"
#include <math.h>
#include <stdbool.h>
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t matmul(void* args, int32_t* arg_type_ids, int32_t num_args, void* out_ret_value, int32_t* out_ret_tcode, void* resource_handle);
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t matmul(void* args, int32_t* arg_type_ids, int32_t num_args, void* out_ret_value, int32_t* out_ret_tcode, void* resource_handle) {
  int32_t _15__code = arg_type_ids[0];
  int32_t _16__code = arg_type_ids[1];
  int32_t C_code = arg_type_ids[2];
  void* _15_ = (((TVMValue*)args)[0].v_handle);
  void* _16_ = (((TVMValue*)args)[1].v_handle);
  void* C = (((TVMValue*)args)[2].v_handle);
  void* matmul__15__shape = (((DLTensor*)_15_)[0].shape);
  void* matmul__15__strides = (((DLTensor*)_15_)[0].strides);
  int32_t dev_id = (((DLTensor*)_15_)[0].device.device_id);
  void